# Section VI Enablers Evidence Lab (Full-Scan + JSON/Markdown Fusion)

This notebook builds the evidence layer for Section VI (Enabling Technologies and Hardware).
- Full scan: processed markdowns + O_ISAC JSON
- Variant-aware retrieval (lexical + fuzzy + LLM entailment)
- Two-model flow with escalation (pass-1 fast, pass-2 strict)
- Section VI metrics only: OPA/RIS structured metrics + PIC/ML/photonic-generation coverage
- Section II/IV safe: no metric-plane mixing; medium labels normalized to Section IV taxonomy

Usage:
- Tune model and rate settings in `# @title 3. Config`.
- Keep `RESUME=True`; checkpoints are under `analysis/VI_ev_v2/checkpoints`.


In [1]:
# @title 1. Install Dependencies
!pip install -q groq rapidfuzz tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.1 MB/s eta 0:00:00


In [2]:
# @title 2. Setup & Mount Drive
from google.colab import drive, userdata
import os, re, json, glob
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print('Working dir:', os.getcwd())
else:
    print('Path not found:', BASE_DIR)


Mounted at /content/drive
Working dir: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST


In [3]:
# @title 3. Config
PROCESSED_MD_DIR = Path('data/proc_markdowns')
JSON_DIR = Path('data/ext_res_v4')
UNIFIED_JSON = JSON_DIR / 'extraction_v4_unified.json'
OUTPUT_DIR = Path('analysis/VI_ev_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_PROFILE = 'FULL_RESCAN'
NORMALIZE_LABELS = True
TARGET_PAPERS = None
LIMIT = None
LLM_CALLS = True

MODEL_PASS1 = 'meta-llama/llama-4-scout-17b-16e-instruct'
MODEL_PASS2 = 'llama-3.3-70b-versatile'
MODEL_VARIANT_GEN = MODEL_PASS1
USE_ESCALATION = True
ESCALATE_LABELS = {'INDIRECT', 'NONE', 'WEAK'}

RPM_BY_MODEL = {MODEL_PASS1: 120, MODEL_PASS2: 40, MODEL_VARIANT_GEN: 120}
DEFAULT_RPM = 30
MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2.0

MAX_VARIANTS_PER_CONCEPT = 12
MAX_HITS_PER_CONCEPT_PER_PAPER = 6
MAX_CONTEXT_CHARS = 1200
CLASSIFY_CHUNK_SIZE = 4
BATCH_SIZE_PAPERS = 10

RESUME = True
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_MAP_PATH = Path('analysis/II_sch_map.md')
GOV_PATH = Path('analysis/II_met_gov.md')
IV_AXIS_PATH = Path('analysis/IV_ev_v2/axis_definitions.md')
IV_MAP_PATH = Path('analysis/IV_ev_v2/mapping_rules.md')

schema_text = SCHEMA_MAP_PATH.read_text(encoding='utf-8', errors='ignore') if SCHEMA_MAP_PATH.exists() else ''
gov_text = GOV_PATH.read_text(encoding='utf-8', errors='ignore') if GOV_PATH.exists() else ''
iv_axis_text = IV_AXIS_PATH.read_text(encoding='utf-8', errors='ignore') if IV_AXIS_PATH.exists() else ''
iv_map_text = IV_MAP_PATH.read_text(encoding='utf-8', errors='ignore') if IV_MAP_PATH.exists() else ''

print('Config ready. Output:', OUTPUT_DIR)
print('Run profile:', RUN_PROFILE)
print('Pass-1 model:', MODEL_PASS1)
print('Pass-2 model:', MODEL_PASS2)
print('Section II schema loaded:', bool(schema_text))
print('Section II governance loaded:', bool(gov_text))
print('Section IV axis loaded:', bool(iv_axis_text))
print('Section IV mapping loaded:', bool(iv_map_text))


Config ready. Output: analysis/VI_ev_v2
Run profile: FULL_RESCAN
Pass-1 model: meta-llama/llama-4-scout-17b-16e-instruct
Pass-2 model: llama-3.3-70b-versatile
Section II schema loaded: True
Section II governance loaded: True
Section IV axis loaded: True
Section IV mapping loaded: True


In [4]:
# @title 4. Load O_ISAC JSON Index
def load_json_index(json_dir: Path):
    index = {}
    for p in sorted(json_dir.glob('O_ISAC_*_v4.json')):
        paper_id = p.stem.replace('_v4','')
        try:
            index[paper_id] = json.loads(p.read_text(encoding='utf-8', errors='ignore'))
        except Exception as e:
            index[paper_id] = {'_error': str(e)}
    unified = None
    if UNIFIED_JSON.exists():
        unified = json.loads(UNIFIED_JSON.read_text(encoding='utf-8', errors='ignore'))
    return index, unified

json_index, unified_json = load_json_index(JSON_DIR)
print('JSON files loaded:', len(json_index))
print('Unified JSON:', 'yes' if unified_json else 'no')


JSON files loaded: 221
Unified JSON: yes


In [5]:
# @title 5. Load Processed Markdowns (Canonical per paper)
def canonical_md_path(paths, paper_id):
    scored = []
    for p in paths:
        p = Path(p)
        score = 0
        if (p.parent / 'visual_analysis.txt').exists():
            score += 3
        if p.parent.name == paper_id and p.parent.parent.name == paper_id:
            score += 2
        score += len(p.parts) * 0.1
        scored.append((score, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1] if scored else None

def load_processed_markdowns(target_ids=None, limit=None):
    search_path = PROCESSED_MD_DIR
    all_files = list(search_path.rglob('*.md'))
    md_files = [p for p in all_files if 'O_ISAC_' in p.name]

    grouped = {}
    for p in md_files:
        m = re.search(r'(O_ISAC_\d+)', p.name)
        if not m:
            continue
        paper_id = m.group(1)
        if target_ids and paper_id not in target_ids:
            continue
        grouped.setdefault(paper_id, []).append(p)

    records = []
    for i, (paper_id, paths) in enumerate(sorted(grouped.items())):
        if limit and i >= limit:
            break
        canon = canonical_md_path(paths, paper_id)
        if not canon:
            continue
        text = canon.read_text(encoding='utf-8', errors='ignore')
        lines = text.splitlines()
        va_path = canon.parent / 'visual_analysis.txt'
        va_text = va_path.read_text(encoding='utf-8', errors='ignore') if va_path.exists() else ''
        records.append({
            'paper_id': paper_id,
            'md_path': str(canon),
            'text': text,
            'lines': lines,
            'visual_analysis': va_text
        })
    return records

papers = load_processed_markdowns(target_ids=TARGET_PAPERS, limit=LIMIT)
print('Markdown papers loaded:', len(papers))


Markdown papers loaded: 221


In [6]:
# @title 6. Heading + Enabler Helpers
import re

MEDIUM_ALIAS_MAP = {
    'visible_light': 'wireless_vlc',
    'vlc': 'wireless_vlc',
    'rf': 'wireless_rf',
    'photo_thz': 'terahertz',
    'photonic_thz': 'terahertz',
}

def build_heading_map(lines):
    current = []
    heading_map = {}
    for i, line in enumerate(lines):
        if line.startswith('#'):
            level = len(line) - len(line.lstrip('#'))
            title = line.strip('#').strip()
            if level <= len(current):
                current = current[:level - 1]
            current.append(title)
        heading_map[i] = ' > '.join(current) if current else 'no_heading'
    return heading_map

def get_context(lines, idx, window=2):
    start = max(0, idx - window)
    end = min(len(lines), idx + window + 1)
    return '\n'.join(lines[start:end])

def to_float(x):
    try:
        if x is None:
            return None
        if isinstance(x, str) and not x.strip():
            return None
        return float(x)
    except Exception:
        return None

def normalize_token(value):
    s = str(value or '').strip().lower()
    if not s or s in {'nr', 'not reported', 'nan', 'none', 'na', 'n/a'}:
        return 'unknown'
    s = s.replace('/', '_')
    s = re.sub(r'[^a-z0-9_\-\s]+', '', s)
    s = re.sub(r'[\s\-]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s or 'unknown'

def normalize_medium_label(value):
    s = normalize_token(value)
    if s == 'unknown':
        return s
    return MEDIUM_ALIAS_MAP.get(s, s)

def get_record_medium(record):
    if not isinstance(record, dict):
        return 'unknown'
    clsf = record.get('study_level', {}).get('classification', {})
    if not isinstance(clsf, dict):
        return 'unknown'
    return normalize_medium_label(clsf.get('oisac_medium_class', 'unknown'))

def get_scenarios(record):
    if not isinstance(record, dict):
        return []
    sl = record.get('scenario_level', [])
    if isinstance(sl, list):
        return [x for x in sl if isinstance(x, dict)]
    if isinstance(sl, dict):
        return [sl]
    return []

def normalize_ris_type(value):
    s = normalize_token(value)
    alias = {'optical_ris': 'oris', 'optical_r_i_s': 'oris', 'oris': 'oris', 'ris': 'ris'}
    return alias.get(s, s)

def extract_enabler_details(scn):
    ed = scn.get('enabling_tech_details', {}) if isinstance(scn, dict) else {}
    if not isinstance(ed, dict):
        ed = {}
    return {
        'opa_num_emitters': to_float(ed.get('opa_num_emitters')),
        'opa_num_elements': to_float(ed.get('opa_num_elements')),
        'opa_steering_range_deg': to_float(ed.get('opa_steering_range_deg')),
        'opa_beamwidth_deg': to_float(ed.get('opa_beamwidth_deg')),
        'ris_num_elements': to_float(ed.get('ris_num_elements')),
        'ris_phase_bits': to_float(ed.get('ris_phase_bits')),
        'ris_type': normalize_ris_type(ed.get('ris_type')),
    }


In [7]:
# @title 7. Groq Client + Variant Generator (Cache + Per-Model Rate Limit)
from groq import Groq
from collections import deque
import time
import random

VARIANT_CACHE = OUTPUT_DIR / 'variant_cache.json'
if VARIANT_CACHE.exists():
    variant_cache = json.loads(VARIANT_CACHE.read_text(encoding='utf-8'))
else:
    variant_cache = {}

_GROQ_CLIENT = None
REQUEST_LOG_BY_MODEL = {}


def get_groq_client():
    global _GROQ_CLIENT
    if _GROQ_CLIENT is not None:
        return _GROQ_CLIENT

    try:
        api_key = userdata.get('GROQ_API_KEY')
    except Exception:
        api_key = os.environ.get('GROQ_API_KEY')

    if not api_key:
        raise ValueError('GROQ_API_KEY not found in Colab Secrets or env.')

    _GROQ_CLIENT = Groq(api_key=api_key)
    return _GROQ_CLIENT


def get_model_rpm(model_name):
    return RPM_BY_MODEL.get(model_name, DEFAULT_RPM)


def throttle_requests(model_name):
    rpm = get_model_rpm(model_name)
    if rpm <= 0:
        return

    q = REQUEST_LOG_BY_MODEL.setdefault(model_name, deque())
    now = time.time()

    while q and now - q[0] > 60:
        q.popleft()

    if len(q) >= rpm:
        wait_s = 60 - (now - q[0]) + 0.1
        wait_s = max(wait_s, 0.1)
        print(f'Rate limit guard ({model_name}): sleeping {wait_s:.1f}s')
        time.sleep(wait_s)
        now = time.time()
        while q and now - q[0] > 60:
            q.popleft()

    q.append(time.time())


def safe_chat_completion(model_name, messages, expect_json=False, temperature=0.2):
    if not LLM_CALLS:
        return None

    client = get_groq_client()

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            throttle_requests(model_name)
            kwargs = {
                'model': model_name,
                'messages': messages,
                'temperature': temperature
            }
            if expect_json:
                kwargs['response_format'] = {'type': 'json_object'}

            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            if attempt >= MAX_RETRIES:
                print(f'LLM call failed ({model_name}) after {MAX_RETRIES} attempts: {e}')
                return None
            sleep_s = RETRY_BASE_SECONDS * (2 ** (attempt - 1)) + random.uniform(0.0, 0.5)
            print(f'LLM retry ({model_name}) {attempt}/{MAX_RETRIES}: {e}; sleeping {sleep_s:.1f}s')
            time.sleep(sleep_s)


def get_variants(concept):
    if concept in variant_cache:
        vals = variant_cache[concept]
        return vals[:MAX_VARIANTS_PER_CONCEPT]

    if not LLM_CALLS:
        vals = [concept]
        variant_cache[concept] = vals
        return vals

    system_prompt = (
        'You generate lexical variants and paraphrases for evidence retrieval. '
        'Return compact JSON: {"variants": ["..."]}.'
    )
    user_prompt = (
        f'Concept: {concept}\n'
        'Return up to 12 variants including synonyms, abbreviations, paraphrases, and morphological forms. '
        'Keep each variant short.'
    )

    content = safe_chat_completion(
        model_name=MODEL_VARIANT_GEN,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        expect_json=True,
        temperature=0.2
    )

    variants = [concept]
    if content:
        try:
            data = json.loads(content)
            llm_vars = data.get('variants', [])
            if isinstance(llm_vars, list):
                for item in llm_vars:
                    if isinstance(item, str) and item.strip():
                        variants.append(item.strip())
        except Exception:
            pass

    dedup = []
    seen = set()
    for v in variants:
        key = v.lower().strip()
        if not key or key in seen:
            continue
        seen.add(key)
        dedup.append(v)

    dedup = dedup[:MAX_VARIANTS_PER_CONCEPT]
    variant_cache[concept] = dedup
    VARIANT_CACHE.write_text(json.dumps(variant_cache, ensure_ascii=False, indent=2), encoding='utf-8')
    return dedup


In [8]:
# @title 8. Retrieval + Entailment Classification (Two-Model, Batched, Checkpointed)
def scan_lines_for_variants(lines, variants, fuzzy_threshold=85):
    hits = []
    for i, line in enumerate(lines):
        text = line.strip()
        if not text:
            continue
        low = text.lower()
        for v in variants:
            vlow = v.lower()
            if vlow in low:
                hits.append((i, line, v, 'lexical'))
                break
            score = fuzz.partial_ratio(vlow, low)
            if score >= fuzzy_threshold:
                hits.append((i, line, v, f'fuzzy:{score}'))
                break
    return hits


def clip_text(text, max_chars=MAX_CONTEXT_CHARS):
    if text is None:
        return ''
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + ' ...'


def chunk_list(items, n):
    for i in range(0, len(items), n):
        yield items[i:i+n]


def parse_batch_results(content, n):
    fallback = [{'label': 'WEAK', 'rationale': 'LLM parse failed'} for _ in range(n)]
    if not content:
        return fallback

    try:
        parsed = json.loads(content)
        results = parsed.get('results', [])
        mapped = {int(r['idx']): r for r in results if isinstance(r, dict) and 'idx' in r}
        out = []
        for i in range(n):
            r = mapped.get(i)
            if not r:
                out.append({'label': 'WEAK', 'rationale': 'No label'})
                continue
            label = str(r.get('label', 'WEAK')).upper().strip()
            if label not in {'DIRECT', 'INDIRECT', 'NONE'}:
                label = 'WEAK'
            out.append({'label': label, 'rationale': str(r.get('rationale', ''))})
        return out
    except Exception:
        return fallback


def classify_with_model(concept, contexts, model_name, hint_labels=None):
    compact = []
    for i, ctx in enumerate(contexts):
        row = {'idx': i, 'context': clip_text(ctx)}
        if hint_labels and i < len(hint_labels):
            row['hint_label'] = hint_labels[i]
        compact.append(row)

    system_prompt = (
        'You are an evidence auditor. '
        'For each snippet, decide if the concept is DIRECT, INDIRECT, or NONE. '
        'Return strict JSON object with key "results": '
        '[{"idx":0,"label":"DIRECT|INDIRECT|NONE","rationale":"..."}]'
    )
    user_prompt = (
        f'Concept: {concept}\n'
        f'Snippets JSON:\n{json.dumps(compact, ensure_ascii=False)}'
    )

    content = safe_chat_completion(
        model_name=model_name,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        expect_json=True,
        temperature=0.1
    )

    return parse_batch_results(content, len(contexts))


def classify_hits_batch(concept, contexts):
    if not contexts:
        return []

    if not LLM_CALLS:
        return [{
            'label': 'WEAK',
            'rationale': 'LLM disabled',
            'label_pass1': 'WEAK',
            'rationale_pass1': 'LLM disabled',
            'model_pass1': MODEL_PASS1,
            'label_pass2': '',
            'rationale_pass2': '',
            'model_pass2': '',
            'escalated': False
        } for _ in contexts]

    pass1 = classify_with_model(concept, contexts, MODEL_PASS1)
    out = []
    for r in pass1:
        out.append({
            'label': r.get('label', 'WEAK'),
            'rationale': r.get('rationale', ''),
            'label_pass1': r.get('label', 'WEAK'),
            'rationale_pass1': r.get('rationale', ''),
            'model_pass1': MODEL_PASS1,
            'label_pass2': '',
            'rationale_pass2': '',
            'model_pass2': '',
            'escalated': False
        })

    if USE_ESCALATION:
        idxs = [i for i, r in enumerate(out) if r['label'] in ESCALATE_LABELS]
        if idxs:
            sub_contexts = [contexts[i] for i in idxs]
            hints = [out[i]['label_pass1'] for i in idxs]
            pass2 = classify_with_model(concept, sub_contexts, MODEL_PASS2, hint_labels=hints)

            for j, i in enumerate(idxs):
                r2 = pass2[j]
                out[i]['label_pass2'] = r2.get('label', 'WEAK')
                out[i]['rationale_pass2'] = r2.get('rationale', '')
                out[i]['model_pass2'] = MODEL_PASS2
                out[i]['escalated'] = True

                if r2.get('label') in {'DIRECT', 'INDIRECT', 'NONE'}:
                    out[i]['label'] = r2.get('label')
                    out[i]['rationale'] = r2.get('rationale', '')

    return out


def classify_hits_chunked(concept, contexts):
    out = []
    for chunk in chunk_list(contexts, CLASSIFY_CHUNK_SIZE):
        out.extend(classify_hits_batch(concept, chunk))
    return out


def append_rows_csv(out_csv, rows):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    if out_csv.exists():
        df_new.to_csv(out_csv, mode='a', header=False, index=False)
    else:
        df_new.to_csv(out_csv, index=False)


def checkpoint_path(section_name):
    return CHECKPOINT_DIR / f'{section_name}_done_ids.json'


def load_done_ids(section_name):
    if not RESUME:
        return set()
    cp = checkpoint_path(section_name)
    if not cp.exists():
        return set()
    try:
        data = json.loads(cp.read_text(encoding='utf-8'))
        return set(data)
    except Exception:
        return set()


def save_done_ids(section_name, done_ids):
    cp = checkpoint_path(section_name)
    cp.write_text(json.dumps(sorted(list(done_ids)), ensure_ascii=False, indent=2), encoding='utf-8')


def process_in_batches(records):
    for batch in chunk_list(records, BATCH_SIZE_PAPERS):
        yield batch


def llm_fields_from_cls(cls):
    return {
        'llm_model_pass1': cls.get('model_pass1', ''),
        'llm_label_pass1': cls.get('label_pass1', ''),
        'llm_model_pass2': cls.get('model_pass2', ''),
        'llm_label_pass2': cls.get('label_pass2', ''),
        'llm_escalated': cls.get('escalated', False),
    }


In [9]:
# @title 9. Section 6A Evidence (Enabling Technologies)
section_name = 'section6A'
out_csv = OUTPUT_DIR / f'{section_name}_evidence.csv'

done_ids = load_done_ids(section_name)
pending = [p for p in papers if p['paper_id'] not in done_ids]
print(f'{section_name}: pending papers = {len(pending)}')

enabler_concepts = {
    'pic': ['photonic integrated circuit', 'integrated photonics', 'silicon photonics', 'photonic integration'],
    'opa': ['optical phased array', 'opa', 'phased array', 'beam steering'],
    'ris': ['optical ris', 'oris', 'reconfigurable intelligent surface', 'intelligent reflecting surface', 'metasurface'],
    'ml': ['machine learning', 'deep learning', 'neural network', 'cnn', 'rnn', 'transformer', 'reinforcement learning'],
    'photonic_generation': ['photonic thz', 'photonic generation', 'photomixing', 'optical heterodyne', 'frequency comb'],
    'programmable_photonics': ['programmable photonics', 'reconfigurable photonic', 'programmable optical front-end'],
}

concept_variants = {}
for concept, terms in enabler_concepts.items():
    expanded = []
    for term in terms:
        expanded.extend(get_variants(term))
    dedup = []
    seen = set()
    for e in expanded:
        key = e.lower().strip()
        if key and key not in seen:
            seen.add(key)
            dedup.append(e)
    concept_variants[concept] = dedup[:MAX_VARIANTS_PER_CONCEPT]

for batch in process_in_batches(pending):
    batch_rows = []
    for paper in tqdm(batch, desc=f'{section_name} batch'):
        paper_id = paper['paper_id']
        lines = paper['lines']
        heading_map = build_heading_map(lines)
        record = json_index.get(paper_id, {})
        medium = get_record_medium(record)

        for concept, variants in concept_variants.items():
            hits = scan_lines_for_variants(lines, variants)
            hits = hits[:MAX_HITS_PER_CONCEPT_PER_PAPER]
            contexts = [get_context(lines, idx) for idx, _, _, _ in hits]
            cls_all = classify_hits_chunked(concept, contexts)

            for (hit, cls) in zip(hits, cls_all):
                idx, line, variant, match_type = hit
                batch_rows.append({
                    'paper_id': paper_id,
                    'section': '6A',
                    'concept': concept,
                    'variant': variant,
                    'match_type': match_type,
                    'strength': cls.get('label', 'WEAK'),
                    'rationale': cls.get('rationale', ''),
                    'quote': line.strip(),
                    'line_start': idx + 1,
                    'line_end': idx + 1,
                    'heading_path': heading_map.get(idx, 'no_heading'),
                    'json_path': '',
                    'json_value': '',
                    'medium': medium,
                    **llm_fields_from_cls(cls),
                })

        if isinstance(record, dict):
            scenarios = get_scenarios(record)
            for sidx, scn in enumerate(scenarios):
                ed = extract_enabler_details(scn)
                for k, v in ed.items():
                    if v is None or v == 'unknown':
                        continue
                    batch_rows.append({
                        'paper_id': paper_id,
                        'section': '6A',
                        'concept': f'json:{k}',
                        'variant': '',
                        'match_type': 'json',
                        'strength': 'DIRECT',
                        'rationale': 'Structured enabling technology field',
                        'quote': '',
                        'line_start': '',
                        'line_end': '',
                        'heading_path': '',
                        'json_path': f'scenario_level[{sidx}].enabling_tech_details.{k}',
                        'json_value': str(v),
                        'medium': medium,
                        'llm_model_pass1': 'json',
                        'llm_label_pass1': 'DIRECT',
                        'llm_model_pass2': '',
                        'llm_label_pass2': '',
                        'llm_escalated': False,
                    })

        done_ids.add(paper_id)

    append_rows_csv(out_csv, batch_rows)
    save_done_ids(section_name, done_ids)
    print(f'{section_name}: wrote {len(batch_rows)} rows; done={len(done_ids)}')

print('Saved:', out_csv)


section6A: pending papers = 221


section6A batch:  60%|██████    | 6/10 [00:43<00:31,  7.84s/it]

Rate limit guard (llama-3.3-70b-versatile): sleeping 15.4s


section6A batch: 100%|██████████| 10/10 [01:34<00:00,  9.40s/it]


section6A: wrote 224 rows; done=10


...

section6A: wrote 239 rows; done=220


section6A batch:   0%|          | 0/1 [00:00<?, ?it/s]

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.5s
Rate limit guard (llama-3.3-70b-versatile): sleeping 0.2s
Rate limit guard (llama-3.3-70b-versatile): sleeping 0.3s
Rate limit guard (llama-3.3-70b-versatile): sleeping 7.0s
Rate limit guard (llama-3.3-70b-versatile): sleeping 0.2s


section6A batch: 100%|██████████| 1/1 [00:15<00:00, 15.74s/it]

section6A: wrote 26 rows; done=221
Saved: analysis/VI_ev_v2/section6A_evidence.csv


In [10]:
# @title 10. Section 6B Evidence (OPA Metrics)
rows = []
for paper in tqdm(papers, desc='section6B'):
    paper_id = paper['paper_id']
    record = json_index.get(paper_id, {})
    if not isinstance(record, dict):
        continue

    medium = get_record_medium(record)
    scenarios = get_scenarios(record)
    for sidx, scn in enumerate(scenarios):
        ed = extract_enabler_details(scn)
        has_any = any(ed.get(k) is not None for k in ['opa_num_emitters', 'opa_num_elements', 'opa_steering_range_deg', 'opa_beamwidth_deg'])
        if not has_any:
            continue

        rows.append({
            'paper_id': paper_id,
            'medium': medium,
            'scenario_index': sidx,
            'opa_num_emitters': ed.get('opa_num_emitters'),
            'opa_num_elements': ed.get('opa_num_elements'),
            'opa_steering_range_deg': ed.get('opa_steering_range_deg'),
            'opa_beamwidth_deg': ed.get('opa_beamwidth_deg'),
        })

df = pd.DataFrame(rows)
out_csv = OUTPUT_DIR / 'section6B_opa_metrics.csv'
df.to_csv(out_csv, index=False)
print('Saved:', out_csv, 'rows=', len(df))


section6B: 100%|██████████| 221/221 [00:00<00:00, 33832.44it/s]

Saved: analysis/VI_ev_v2/section6B_opa_metrics.csv rows= 122


In [11]:
# @title 11. Section 6C Evidence (RIS Metrics)
rows = []
for paper in tqdm(papers, desc='section6C'):
    paper_id = paper['paper_id']
    record = json_index.get(paper_id, {})
    if not isinstance(record, dict):
        continue

    medium = get_record_medium(record)
    scenarios = get_scenarios(record)
    for sidx, scn in enumerate(scenarios):
        ed = extract_enabler_details(scn)
        has_any = (ed.get('ris_num_elements') is not None or ed.get('ris_phase_bits') is not None or ed.get('ris_type') not in (None, 'unknown'))
        if not has_any:
            continue

        rows.append({
            'paper_id': paper_id,
            'medium': medium,
            'scenario_index': sidx,
            'ris_num_elements': ed.get('ris_num_elements'),
            'ris_phase_bits': ed.get('ris_phase_bits'),
            'ris_type': ed.get('ris_type'),
        })

df = pd.DataFrame(rows)
out_csv = OUTPUT_DIR / 'section6C_ris_metrics.csv'
df.to_csv(out_csv, index=False)
print('Saved:', out_csv, 'rows=', len(df))


section6C: 100%|██████████| 221/221 [00:00<00:00, 73924.65it/s]

Saved: analysis/VI_ev_v2/section6C_ris_metrics.csv rows= 125


In [12]:
# @title 12. Section 6D Evidence (ML/AI Mentions)
section_name = 'section6D'
out_csv = OUTPUT_DIR / f'{section_name}_evidence.csv'

done_ids = load_done_ids(section_name)
pending = [p for p in papers if p['paper_id'] not in done_ids]
print(f'{section_name}: pending papers = {len(pending)}')

ml_terms = [
    'machine learning', 'deep learning', 'neural network', 'cnn', 'rnn',
    'transformer', 'reinforcement learning', 'supervised learning', 'unsupervised learning',
    'svm', 'random forest', 'gaussian process', 'bayesian optimization'
]

ml_variants = []
for term in ml_terms:
    ml_variants.extend(get_variants(term))

dedup = []
seen = set()
for e in ml_variants:
    key = e.lower().strip()
    if key and key not in seen:
        seen.add(key)
        dedup.append(e)
ml_variants = dedup[:MAX_VARIANTS_PER_CONCEPT]

for batch in process_in_batches(pending):
    batch_rows = []
    for paper in tqdm(batch, desc=f'{section_name} batch'):
        paper_id = paper['paper_id']
        lines = paper['lines']
        heading_map = build_heading_map(lines)
        record = json_index.get(paper_id, {})
        medium = get_record_medium(record)

        hits = scan_lines_for_variants(lines, ml_variants)
        hits = hits[:MAX_HITS_PER_CONCEPT_PER_PAPER]
        contexts = [get_context(lines, idx) for idx, _, _, _ in hits]
        cls_all = classify_hits_chunked('ml', contexts)

        for (hit, cls) in zip(hits, cls_all):
            idx, line, variant, match_type = hit
            batch_rows.append({
                'paper_id': paper_id,
                'section': '6D',
                'concept': 'ml',
                'variant': variant,
                'match_type': match_type,
                'strength': cls.get('label', 'WEAK'),
                'rationale': cls.get('rationale', ''),
                'quote': line.strip(),
                'line_start': idx + 1,
                'line_end': idx + 1,
                'heading_path': heading_map.get(idx, 'no_heading'),
                'json_path': '',
                'json_value': '',
                'medium': medium,
                **llm_fields_from_cls(cls),
            })

        done_ids.add(paper_id)

    append_rows_csv(out_csv, batch_rows)
    save_done_ids(section_name, done_ids)
    print(f'{section_name}: wrote {len(batch_rows)} rows; done={len(done_ids)}')

print('Saved:', out_csv)


section6D: pending papers = 221


section6D batch:  20%|██        | 2/10 [00:04<00:16,  2.06s/it]

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.4s


section6D batch:  30%|███       | 3/10 [00:07<00:16,  2.42s/it]

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.7s


...

section6D: wrote 60 rows; done=220


section6D batch:   0%|          | 0/1 [00:00<?, ?it/s]

Rate limit guard (llama-3.3-70b-versatile): sleeping 0.2s


section6D batch: 100%|██████████| 1/1 [00:02<00:00,  2.20s/it]

section6D: wrote 6 rows; done=221
Saved: analysis/VI_ev_v2/section6D_evidence.csv


In [13]:
# @title 13. Section 6E Summary Outputs (Coverage + Medium Slices)
a_csv = OUTPUT_DIR / 'section6A_evidence.csv'
b_csv = OUTPUT_DIR / 'section6B_opa_metrics.csv'
c_csv = OUTPUT_DIR / 'section6C_ris_metrics.csv'
d_csv = OUTPUT_DIR / 'section6D_evidence.csv'

a = pd.read_csv(a_csv) if a_csv.exists() else pd.DataFrame()
b = pd.read_csv(b_csv) if b_csv.exists() else pd.DataFrame()
c = pd.read_csv(c_csv) if c_csv.exists() else pd.DataFrame()
d = pd.read_csv(d_csv) if d_csv.exists() else pd.DataFrame()

def supported_ids(df, concept_name):
    if df.empty or 'concept' not in df.columns or 'paper_id' not in df.columns:
        return set()
    sub = df[df['concept'].astype(str) == concept_name].copy()
    if sub.empty:
        return set()
    out = set()
    for pid, grp in sub.groupby('paper_id'):
        direct = (grp['strength'].astype(str).str.upper() == 'DIRECT').sum() if 'strength' in grp.columns else 0
        indirect = (grp['strength'].astype(str).str.upper() == 'INDIRECT').sum() if 'strength' in grp.columns else 0
        if direct >= 1 or indirect >= 2:
            out.add(str(pid))
    return out

all_paper_ids = sorted([p.get('paper_id') for p in papers])
n_total_papers = len(all_paper_ids)

opa_papers = set(b['paper_id'].astype(str)) if not b.empty and 'paper_id' in b.columns else set()
ris_papers = set(c['paper_id'].astype(str)) if not c.empty and 'paper_id' in c.columns else set()
pic_papers = supported_ids(a, 'pic')
pg_papers = supported_ids(a, 'photonic_generation')
pp_papers = supported_ids(a, 'programmable_photonics')
ml_papers = supported_ids(a, 'ml') | supported_ids(d, 'ml')

summary_rows = [{
    'n_total_papers': int(n_total_papers),
    'n_opa_papers': int(len(opa_papers)),
    'n_ris_papers': int(len(ris_papers)),
    'n_pic_papers': int(len(pic_papers)),
    'n_ml_papers': int(len(ml_papers)),
    'n_photonic_generation_papers': int(len(pg_papers)),
    'n_programmable_photonics_papers': int(len(pp_papers)),
    'median_opa_emitters': float(b['opa_num_emitters'].median()) if 'opa_num_emitters' in b.columns and len(b) else None,
    'median_opa_beamwidth_deg': float(b['opa_beamwidth_deg'].median()) if 'opa_beamwidth_deg' in b.columns and len(b) else None,
    'median_ris_elements': float(c['ris_num_elements'].median()) if 'ris_num_elements' in c.columns and len(c) else None,
    'median_ris_phase_bits': float(c['ris_phase_bits'].median()) if 'ris_phase_bits' in c.columns and len(c) else None,
}]

summary_csv = OUTPUT_DIR / 'section6E_summary_table.csv'
pd.DataFrame(summary_rows).to_csv(summary_csv, index=False)
summary_json = OUTPUT_DIR / 'section6E_summary.json'
summary_json.write_text(json.dumps(summary_rows[0], indent=2), encoding='utf-8')

rows = []
for pid in all_paper_ids:
    rec = json_index.get(pid, {})
    medium = get_record_medium(rec)
    rows.append({
        'paper_id': pid,
        'medium': medium,
        'has_opa': pid in opa_papers,
        'has_ris': pid in ris_papers,
        'has_pic': pid in pic_papers,
        'has_ml': pid in ml_papers,
        'has_photonic_generation': pid in pg_papers,
        'has_programmable_photonics': pid in pp_papers,
    })

medium_df = pd.DataFrame(rows)
medium_slice = medium_df.groupby('medium', dropna=False).agg(
    n_papers=('paper_id', 'nunique'),
    n_has_opa=('has_opa', 'sum'),
    n_has_ris=('has_ris', 'sum'),
    n_has_pic=('has_pic', 'sum'),
    n_has_ml=('has_ml', 'sum'),
    n_has_photonic_generation=('has_photonic_generation', 'sum'),
    n_has_programmable_photonics=('has_programmable_photonics', 'sum'),
).reset_index().sort_values('n_papers', ascending=False)

medium_csv = OUTPUT_DIR / 'section6E_medium_slices.csv'
medium_slice.to_csv(medium_csv, index=False)

print('Saved:', summary_csv)
print('Saved:', summary_json)
print('Saved:', medium_csv)


Saved: analysis/VI_ev_v2/section6E_summary_table.csv
Saved: analysis/VI_ev_v2/section6E_summary.json
Saved: analysis/VI_ev_v2/section6E_medium_slices.csv


In [14]:
# @title 14. Post-Processing Artifacts (Section 6)
import hashlib
from collections import defaultdict

def as_int_or_blank(x):
    try:
        if pd.isna(x) or x == '':
            return ''
        return int(float(x))
    except Exception:
        return ''

a_csv = OUTPUT_DIR / 'section6A_evidence.csv'
d_csv = OUTPUT_DIR / 'section6D_evidence.csv'
b_csv = OUTPUT_DIR / 'section6B_opa_metrics.csv'
c_csv = OUTPUT_DIR / 'section6C_ris_metrics.csv'
a = pd.read_csv(a_csv) if a_csv.exists() else pd.DataFrame()
d = pd.read_csv(d_csv) if d_csv.exists() else pd.DataFrame()
b = pd.read_csv(b_csv) if b_csv.exists() else pd.DataFrame()
c = pd.read_csv(c_csv) if c_csv.exists() else pd.DataFrame()

# retrieval_hits.jsonl
retrieval_path = OUTPUT_DIR / 'retrieval_hits.jsonl'
with retrieval_path.open('w', encoding='utf-8') as f:
    for sec_name, df in [('6A', a), ('6D', d)]:
        if df.empty:
            continue
        for _, r in df.iterrows():
            mt = str(r.get('match_type', ''))
            if mt.lower() == 'json':
                continue
            rec = {
                'paper_id': str(r.get('paper_id', '')), 'section': sec_name,
                'concept': str(r.get('concept', '')), 'variant': str(r.get('variant', '')),
                'match_type': mt, 'quote': str(r.get('quote', '')),
                'line_start': as_int_or_blank(r.get('line_start', '')),
                'line_end': as_int_or_blank(r.get('line_end', '')),
                'heading_path': str(r.get('heading_path', '')),
                'strength': str(r.get('strength', '')),
                'rationale': str(r.get('rationale', '')),
            }
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('Saved:', retrieval_path)

# anchor_table.csv
anchor_rows = []
for sec_name, df in [('6A', a), ('6D', d)]:
    if df.empty:
        continue
    for _, r in df.iterrows():
        concept = str(r.get('concept', '')).strip()
        claim_key = f'{sec_name}|{concept}'
        claim_id = hashlib.sha1(claim_key.encode('utf-8')).hexdigest()[:12]
        anchor_rows.append({
            'claim_id': claim_id, 'claim_key': claim_key, 'section': sec_name,
            'paper_id': str(r.get('paper_id', '')), 'concept': concept,
            'variant': str(r.get('variant', '')), 'match_type': str(r.get('match_type', '')),
            'strength': str(r.get('strength', '')), 'rationale': str(r.get('rationale', '')),
            'quote': str(r.get('quote', '')),
            'line_start': as_int_or_blank(r.get('line_start', '')),
            'line_end': as_int_or_blank(r.get('line_end', '')),
            'heading_path': str(r.get('heading_path', '')),
            'json_path': str(r.get('json_path', '')), 'json_value': str(r.get('json_value', '')),
        })

anchor_cols = ['claim_id','claim_key','section','paper_id','concept','variant','match_type','strength','rationale','quote','line_start','line_end','heading_path','json_path','json_value','claim_supported']
if anchor_rows:
    anc = pd.DataFrame(anchor_rows)
    agg = anc.assign(
        is_direct=anc['strength'].astype(str).str.upper().eq('DIRECT'),
        is_indirect=anc['strength'].astype(str).str.upper().eq('INDIRECT')
    ).groupby('claim_id', as_index=False)[['is_direct', 'is_indirect']].sum()
    agg['claim_supported'] = (agg['is_direct'] >= 1) | (agg['is_indirect'] >= 2)
    anc = anc.merge(agg[['claim_id', 'claim_supported']], on='claim_id', how='left')
else:
    anc = pd.DataFrame(columns=anchor_cols)
anc = anc[anchor_cols]
anchor_path = OUTPUT_DIR / 'anchor_table.csv'
anc.to_csv(anchor_path, index=False)
print('Saved:', anchor_path)

# evidence_graph.jsonl + cluster_map.csv
opa_papers = set(b['paper_id'].astype(str)) if not b.empty and 'paper_id' in b.columns else set()
ris_papers = set(c['paper_id'].astype(str)) if not c.empty and 'paper_id' in c.columns else set()
supported_by_paper = defaultdict(set)
if not anc.empty:
    for _, r in anc.iterrows():
        if bool(r.get('claim_supported', False)):
            supported_by_paper[str(r.get('paper_id', ''))].add(str(r.get('concept', '')))
anchors_by_paper = defaultdict(list)
for _, r in anc.iterrows():
    anchors_by_paper[str(r.get('paper_id', ''))].append({'section': str(r.get('section', '')), 'concept': str(r.get('concept', '')), 'strength': str(r.get('strength', ''))})

graph_path = OUTPUT_DIR / 'evidence_graph.jsonl'
cluster_rows = []
with graph_path.open('w', encoding='utf-8') as f:
    for paper_id, rec in sorted(json_index.items()):
        medium = get_record_medium(rec)
        concepts = supported_by_paper.get(paper_id, set())
        graph_rec = {
            'paper_id': paper_id,
            'structured': {
                'medium': medium,
                'has_opa_structured': paper_id in opa_papers,
                'has_ris_structured': paper_id in ris_papers,
                'has_pic_supported': 'pic' in concepts,
                'has_ml_supported': 'ml' in concepts,
                'has_photonic_generation_supported': 'photonic_generation' in concepts,
                'has_programmable_photonics_supported': 'programmable_photonics' in concepts,
            },
            'anchor_count': len(anchors_by_paper.get(paper_id, [])),
            'anchors': anchors_by_paper.get(paper_id, []),
        }
        f.write(json.dumps(graph_rec, ensure_ascii=False) + '\n')
        conf = 'high' if graph_rec['anchor_count'] >= 8 else ('medium' if graph_rec['anchor_count'] >= 3 else 'low')
        cluster_rows.append({
            'paper_id': paper_id, 'medium': medium,
            'has_opa_structured': graph_rec['structured']['has_opa_structured'],
            'has_ris_structured': graph_rec['structured']['has_ris_structured'],
            'has_pic_supported': graph_rec['structured']['has_pic_supported'],
            'has_ml_supported': graph_rec['structured']['has_ml_supported'],
            'has_photonic_generation_supported': graph_rec['structured']['has_photonic_generation_supported'],
            'has_programmable_photonics_supported': graph_rec['structured']['has_programmable_photonics_supported'],
            'anchor_count': graph_rec['anchor_count'], 'confidence': conf,
        })
cluster_path = OUTPUT_DIR / 'cluster_map.csv'
pd.DataFrame(cluster_rows).to_csv(cluster_path, index=False)
print('Saved:', graph_path)
print('Saved:', cluster_path)

# axis + mapping docs
axis_md = '\n'.join([
    '# Section 6 Axis Definitions (v2)',
    '',
    'Axis-1 Medium: normalized labels aligned with Section IV taxonomy mapping.',
    'Axis-2 Enabler families: PIC, OPA, RIS/ORIS, ML/AI, photonic generation, programmable photonics.',
    'Axis-3 Enabler metrics: OPA emitters/elements/steering/beamwidth; RIS elements/phase bits/type.',
    'Axis-4 Evidence gate: claim_supported = (>=1 DIRECT) OR (>=2 INDIRECT).',
    'Governance note: Section VI does not create OSNR/SNR or resolution/accuracy cross-plane claims.',
])
(OUTPUT_DIR / 'axis_definitions.md').write_text(axis_md, encoding='utf-8')

mapping_md = '\n'.join([
    '# Section 6 Mapping Rules (v2)',
    '',
    '1. JSON enabling_tech_details has priority for OPA/RIS metrics when present.',
    '2. Text anchors support enabler prevalence claims for PIC/ML/photonic-generation families.',
    '3. RIS type labels are normalized (ORIS/RIS/metasurface aliases).',
    '4. Medium normalization follows the same policy used in Section IV outputs.',
    '5. No Section II metric-plane conversion or Delta z/Delta r_min substitution is permitted in Section VI claims.',
])
(OUTPUT_DIR / 'mapping_rules.md').write_text(mapping_md, encoding='utf-8')
print('Saved:', OUTPUT_DIR / 'axis_definitions.md')
print('Saved:', OUTPUT_DIR / 'mapping_rules.md')

# contract_violations.csv (real checks)
violations = []
if not b.empty:
    for _, r in b.iterrows():
        pid = str(r.get('paper_id', ''))
        emitters = to_float(r.get('opa_num_emitters'))
        elements = to_float(r.get('opa_num_elements'))
        steering = to_float(r.get('opa_steering_range_deg'))
        beamwidth = to_float(r.get('opa_beamwidth_deg'))
        if emitters is not None and emitters <= 0:
            violations.append({'paper_id': pid, 'section': '6B', 'category': 'ENABLER_METRIC', 'severity': 'MAJOR', 'reason': 'OPA emitters must be positive', 'evidence': f'opa_num_emitters={emitters}'})
        if elements is not None and elements <= 0:
            violations.append({'paper_id': pid, 'section': '6B', 'category': 'ENABLER_METRIC', 'severity': 'MAJOR', 'reason': 'OPA elements must be positive', 'evidence': f'opa_num_elements={elements}'})
        if steering is not None and steering <= 0:
            violations.append({'paper_id': pid, 'section': '6B', 'category': 'ENABLER_METRIC', 'severity': 'MAJOR', 'reason': 'OPA steering range must be positive', 'evidence': f'opa_steering_range_deg={steering}'})
        if beamwidth is not None and beamwidth <= 0:
            violations.append({'paper_id': pid, 'section': '6B', 'category': 'ENABLER_METRIC', 'severity': 'MAJOR', 'reason': 'OPA beamwidth must be positive', 'evidence': f'opa_beamwidth_deg={beamwidth}'})
if not c.empty:
    for _, r in c.iterrows():
        pid = str(r.get('paper_id', ''))
        elems = to_float(r.get('ris_num_elements'))
        bits = to_float(r.get('ris_phase_bits'))
        rtype = str(r.get('ris_type', 'unknown')).strip().lower()
        if elems is not None and elems <= 0:
            violations.append({'paper_id': pid, 'section': '6C', 'category': 'ENABLER_METRIC', 'severity': 'MAJOR', 'reason': 'RIS element count must be positive', 'evidence': f'ris_num_elements={elems}'})
        if bits is not None and bits < 0:
            violations.append({'paper_id': pid, 'section': '6C', 'category': 'ENABLER_METRIC', 'severity': 'MAJOR', 'reason': 'RIS phase bits cannot be negative', 'evidence': f'ris_phase_bits={bits}'})
        if rtype in {'', 'unknown', 'na', 'nr'} and (elems is not None or bits is not None):
            violations.append({'paper_id': pid, 'section': '6C', 'category': 'ENABLER_INCOMPLETE', 'severity': 'MINOR', 'reason': 'RIS type missing while RIS metrics are reported', 'evidence': f'ris_type={rtype}; ris_num_elements={elems}; ris_phase_bits={bits}'})
if not a.empty:
    text_a = a[a['match_type'].astype(str).str.lower() != 'json'].copy() if 'match_type' in a.columns else a.copy()
    for concept in ['pic', 'photonic_generation', 'programmable_photonics', 'ml']:
        sub = text_a[text_a['concept'].astype(str) == concept] if 'concept' in text_a.columns else pd.DataFrame()
        if sub.empty:
            continue
        for pid, grp in sub.groupby('paper_id'):
            direct = (grp['strength'].astype(str).str.upper() == 'DIRECT').sum() if 'strength' in grp.columns else 0
            indirect = (grp['strength'].astype(str).str.upper() == 'INDIRECT').sum() if 'strength' in grp.columns else 0
            if direct < 1 and indirect < 2:
                violations.append({'paper_id': str(pid), 'section': '6A', 'category': 'EVIDENCE_WEAK', 'severity': 'MINOR', 'reason': f'Enabler concept {concept} lacks support gate', 'evidence': f'direct={direct}; indirect={indirect}'})

viol_df = pd.DataFrame(violations, columns=['paper_id', 'section', 'category', 'severity', 'reason', 'evidence'])
viol_path = OUTPUT_DIR / 'contract_violations.csv'
viol_df.to_csv(viol_path, index=False)
print('Saved:', viol_path, 'rows=', len(viol_df))


Saved: analysis/VI_ev_v2/retrieval_hits.jsonl
Saved: analysis/VI_ev_v2/anchor_table.csv
Saved: analysis/VI_ev_v2/evidence_graph.jsonl
Saved: analysis/VI_ev_v2/cluster_map.csv
Saved: analysis/VI_ev_v2/axis_definitions.md
Saved: analysis/VI_ev_v2/mapping_rules.md
Saved: analysis/VI_ev_v2/contract_violations.csv rows= 687


In [15]:
# @title 15. Readiness Report
report_files = [
    'section6A_evidence.csv',
    'section6B_opa_metrics.csv',
    'section6C_ris_metrics.csv',
    'section6D_evidence.csv',
    'section6E_summary_table.csv',
    'section6E_summary.json',
    'section6E_medium_slices.csv',
    'evidence_graph.jsonl',
    'retrieval_hits.jsonl',
    'anchor_table.csv',
    'axis_definitions.md',
    'mapping_rules.md',
    'cluster_map.csv',
    'contract_violations.csv',
    's6f_dual_view_cmp.csv',
    's6f_dual_view_ex.csv',
    'section6F_dual_view_report.md',
]

report = []
for fname in report_files:
    p = OUTPUT_DIR / fname
    report.append(f"{fname}: " + ('OK' if p.exists() else 'MISSING'))

stats = []
try:
    p6a = OUTPUT_DIR / 'section6A_evidence.csv'
    p6b = OUTPUT_DIR / 'section6B_opa_metrics.csv'
    p6c = OUTPUT_DIR / 'section6C_ris_metrics.csv'
    p6d = OUTPUT_DIR / 'section6D_evidence.csv'
    p6s = OUTPUT_DIR / 'section6E_summary.json'
    pvc = OUTPUT_DIR / 'contract_violations.csv'
    pf = OUTPUT_DIR / 's6f_dual_view_cmp.csv'

    if p6a.exists():
        d = pd.read_csv(p6a)
        stats.append(f"section6A_rows: {len(d)}")
        stats.append(f"section6A_unique_papers: {d['paper_id'].nunique() if 'paper_id' in d.columns else 0}")
    if p6b.exists():
        d = pd.read_csv(p6b)
        stats.append(f"section6B_rows: {len(d)}")
        stats.append(f"section6B_unique_papers: {d['paper_id'].nunique() if 'paper_id' in d.columns else 0}")
    if p6c.exists():
        d = pd.read_csv(p6c)
        stats.append(f"section6C_rows: {len(d)}")
        stats.append(f"section6C_unique_papers: {d['paper_id'].nunique() if 'paper_id' in d.columns else 0}")
    if p6d.exists():
        d = pd.read_csv(p6d)
        stats.append(f"section6D_rows: {len(d)}")
        stats.append(f"section6D_unique_papers: {d['paper_id'].nunique() if 'paper_id' in d.columns else 0}")
    if p6s.exists():
        s = json.loads(p6s.read_text(encoding='utf-8'))
        stats.append(f"n_opa_papers: {s.get('n_opa_papers')}")
        stats.append(f"n_ris_papers: {s.get('n_ris_papers')}")
        stats.append(f"n_pic_papers: {s.get('n_pic_papers')}")
        stats.append(f"n_ml_papers: {s.get('n_ml_papers')}")
        stats.append(f"n_photonic_generation_papers: {s.get('n_photonic_generation_papers')}")
    if pvc.exists():
        v = pd.read_csv(pvc)
        stats.append(f"contract_violations_rows: {len(v)}")

    if pf.exists():
        f = pd.read_csv(pf)
        for _, row in f.iterrows():
            dim = str(row.get('dimension', 'UNK'))
            stats.append(f"dual_{dim}_study_flag_count: {int(row.get('study_flag_count', 0))}")
            stats.append(f"dual_{dim}_raw_metric_count: {int(row.get('raw_metric_count', 0))}")
            stats.append(f"dual_{dim}_strict_metric_count: {int(row.get('strict_metric_count', 0))}")
except Exception as e:
    stats.append(f"stats_error: {e}")

report_path = OUTPUT_DIR / 'readiness_report.md'
report_path.write_text('\n'.join(report + [''] + stats), encoding='utf-8')
print('\n'.join(report + [''] + stats))
print('Saved:', report_path)


section6A_evidence.csv: OK
section6B_opa_metrics.csv: OK
section6C_ris_metrics.csv: OK
section6D_evidence.csv: OK
section6E_summary_table.csv: OK
section6E_summary.json: OK
section6E_medium_slices.csv: OK
evidence_graph.jsonl: OK
retrieval_hits.jsonl: OK
anchor_table.csv: OK
axis_definitions.md: OK
mapping_rules.md: OK
cluster_map.csv: OK
contract_violations.csv: OK
s6f_dual_view_cmp.csv: MISSING
s6f_dual_view_ex.csv: MISSING
section6F_dual_view_report.md: MISSING

section6A_rows: 5045
section6A_unique_papers: 221
section6B_rows: 122
section6B_unique_papers: 121
section6C_rows: 125
section6C_unique_papers: 124
section6D_rows: 1305
section6D_unique_papers: 220
n_opa_papers: 121
n_ris_papers: 124
n_pic_papers: 24
n_ml_papers: 45
n_photonic_generation_papers: 126
contract_violations_rows: 687
Saved: analysis/VI_ev_v2/readiness_report.md


In [16]:
# @title 16. Section 6F Dual-View Comparison (Raw vs Strict)
# This stage DOES NOT overwrite main outputs. It produces an additional comparison package.

b_csv = OUTPUT_DIR / 'section6B_opa_metrics.csv'
c_csv = OUTPUT_DIR / 'section6C_ris_metrics.csv'
a_csv = OUTPUT_DIR / 'section6A_evidence.csv'
d_csv = OUTPUT_DIR / 'section6D_evidence.csv'

a = pd.read_csv(a_csv) if a_csv.exists() else pd.DataFrame()
b = pd.read_csv(b_csv) if b_csv.exists() else pd.DataFrame()
c = pd.read_csv(c_csv) if c_csv.exists() else pd.DataFrame()
d = pd.read_csv(d_csv) if d_csv.exists() else pd.DataFrame()

def truthy(v):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    s = str(v).strip().lower()
    return s in {'1', 'true', 'yes', 'y'}

# Study-level flags from JSON
flag_opa = set()
flag_ris = set()
flag_ml = set()
for pid, rec in sorted(json_index.items()):
    if not isinstance(rec, dict):
        continue
    st = rec.get('study_level', {})
    et = st.get('enabling_tech', {}) if isinstance(st, dict) else {}
    if isinstance(et, dict):
        if truthy(et.get('opa_present')):
            flag_opa.add(pid)
        if truthy(et.get('ris_present')):
            flag_ris.add(pid)
        if truthy(et.get('machine_learning_used')):
            flag_ml.add(pid)

# Raw metric view (current extraction view)
raw_opa = set(b['paper_id'].astype(str)) if not b.empty and 'paper_id' in b.columns else set()
raw_ris = set(c['paper_id'].astype(str)) if not c.empty and 'paper_id' in c.columns else set()

# Strict metric view (positive/valid metrics only)
strict_opa = set()
if not b.empty and 'paper_id' in b.columns:
    bb = b.copy()
    for col in ['opa_num_emitters', 'opa_num_elements', 'opa_steering_range_deg', 'opa_beamwidth_deg']:
        if col in bb.columns:
            bb[col] = pd.to_numeric(bb[col], errors='coerce')
    mask = False
    if 'opa_num_emitters' in bb.columns:
        mask = mask | (bb['opa_num_emitters'] > 0)
    if 'opa_num_elements' in bb.columns:
        mask = mask | (bb['opa_num_elements'] > 0)
    if 'opa_steering_range_deg' in bb.columns:
        mask = mask | (bb['opa_steering_range_deg'] > 0)
    if 'opa_beamwidth_deg' in bb.columns:
        mask = mask | (bb['opa_beamwidth_deg'] > 0)
    strict_opa = set(bb.loc[mask, 'paper_id'].astype(str))

strict_ris = set()
if not c.empty and 'paper_id' in c.columns:
    cc = c.copy()
    for col in ['ris_num_elements', 'ris_phase_bits']:
        if col in cc.columns:
            cc[col] = pd.to_numeric(cc[col], errors='coerce')
    rt = cc['ris_type'].astype(str).str.strip().str.lower() if 'ris_type' in cc.columns else pd.Series([''] * len(cc))
    mask = False
    if 'ris_num_elements' in cc.columns:
        mask = mask | (cc['ris_num_elements'] > 0)
    if 'ris_phase_bits' in cc.columns:
        mask = mask | (cc['ris_phase_bits'] > 0)
    mask = mask | rt.isin({'reflective', 'transmissive', 'hybrid', 'slm_equivalent', 'oris', 'ris'})
    strict_ris = set(cc.loc[mask, 'paper_id'].astype(str))

# ML comparison: study flag vs evidence-supported text
supported_ml = set()
for df in [a, d]:
    if df.empty or 'paper_id' not in df.columns or 'concept' not in df.columns:
        continue
    sub = df[df['concept'].astype(str).eq('ml')]
    if sub.empty:
        continue
    for pid, grp in sub.groupby('paper_id'):
        direct = (grp['strength'].astype(str).str.upper() == 'DIRECT').sum() if 'strength' in grp.columns else 0
        indirect = (grp['strength'].astype(str).str.upper() == 'INDIRECT').sum() if 'strength' in grp.columns else 0
        if direct >= 1 or indirect >= 2:
            supported_ml.add(str(pid))

rows = [
    {
        'dimension': 'OPA',
        'study_flag_count': len(flag_opa),
        'raw_metric_count': len(raw_opa),
        'strict_metric_count': len(strict_opa),
        'flag_intersection_raw': len(flag_opa & raw_opa),
        'flag_intersection_strict': len(flag_opa & strict_opa),
        'raw_only_vs_flag': len(raw_opa - flag_opa),
        'strict_only_vs_flag': len(strict_opa - flag_opa),
    },
    {
        'dimension': 'RIS',
        'study_flag_count': len(flag_ris),
        'raw_metric_count': len(raw_ris),
        'strict_metric_count': len(strict_ris),
        'flag_intersection_raw': len(flag_ris & raw_ris),
        'flag_intersection_strict': len(flag_ris & strict_ris),
        'raw_only_vs_flag': len(raw_ris - flag_ris),
        'strict_only_vs_flag': len(strict_ris - flag_ris),
    },
    {
        'dimension': 'ML',
        'study_flag_count': len(flag_ml),
        'raw_metric_count': len(supported_ml),
        'strict_metric_count': len(supported_ml),
        'flag_intersection_raw': len(flag_ml & supported_ml),
        'flag_intersection_strict': len(flag_ml & supported_ml),
        'raw_only_vs_flag': len(supported_ml - flag_ml),
        'strict_only_vs_flag': len(supported_ml - flag_ml),
    },
]

cmp_df = pd.DataFrame(rows)
cmp_csv = OUTPUT_DIR / 's6f_dual_view_cmp.csv'
cmp_df.to_csv(cmp_csv, index=False)

# Diagnostic paper lists
max_show = 30
paper_rows = [
    {'dimension': 'OPA', 'group': 'flag_only', 'paper_ids': ';'.join(sorted(list(flag_opa - raw_opa))[:max_show])},
    {'dimension': 'OPA', 'group': 'raw_only', 'paper_ids': ';'.join(sorted(list(raw_opa - flag_opa))[:max_show])},
    {'dimension': 'OPA', 'group': 'strict_only', 'paper_ids': ';'.join(sorted(list(strict_opa - flag_opa))[:max_show])},
    {'dimension': 'RIS', 'group': 'flag_only', 'paper_ids': ';'.join(sorted(list(flag_ris - raw_ris))[:max_show])},
    {'dimension': 'RIS', 'group': 'raw_only', 'paper_ids': ';'.join(sorted(list(raw_ris - flag_ris))[:max_show])},
    {'dimension': 'RIS', 'group': 'strict_only', 'paper_ids': ';'.join(sorted(list(strict_ris - flag_ris))[:max_show])},
    {'dimension': 'ML', 'group': 'flag_only', 'paper_ids': ';'.join(sorted(list(flag_ml - supported_ml))[:max_show])},
    {'dimension': 'ML', 'group': 'supported_only', 'paper_ids': ';'.join(sorted(list(supported_ml - flag_ml))[:max_show])},
]
paper_csv = OUTPUT_DIR / 's6f_dual_view_ex.csv'
pd.DataFrame(paper_rows).to_csv(paper_csv, index=False)

# Markdown report
md_lines = []
md_lines.append('# Section 6F Dual-View Comparison')
md_lines.append('')
md_lines.append('This report compares study-level enabling flags vs raw scenario metrics vs strict scenario metrics.')
md_lines.append('It is an additive audit stage and does not block or overwrite Section 6 outputs.')
md_lines.append('')
for _, r in cmp_df.iterrows():
    md_lines.append(f"## {r['dimension']}")
    md_lines.append(f"- study_flag_count: {int(r['study_flag_count'])}")
    md_lines.append(f"- raw_metric_count: {int(r['raw_metric_count'])}")
    md_lines.append(f"- strict_metric_count: {int(r['strict_metric_count'])}")
    md_lines.append(f"- flag_intersection_raw: {int(r['flag_intersection_raw'])}")
    md_lines.append(f"- flag_intersection_strict: {int(r['flag_intersection_strict'])}")
    md_lines.append(f"- raw_only_vs_flag: {int(r['raw_only_vs_flag'])}")
    md_lines.append(f"- strict_only_vs_flag: {int(r['strict_only_vs_flag'])}")
    md_lines.append('')

report_md = OUTPUT_DIR / 'section6F_dual_view_report.md'
report_md.write_text('\n'.join(md_lines), encoding='utf-8')

print('Saved:', cmp_csv)
print('Saved:', paper_csv)
print('Saved:', report_md)
print(cmp_df.to_string(index=False))


Saved: analysis/VI_ev_v2/s6f_dual_view_cmp.csv
Saved: analysis/VI_ev_v2/s6f_dual_view_ex.csv
Saved: analysis/VI_ev_v2/section6F_dual_view_report.md
dimension  study_flag_count  raw_metric_count  strict_metric_count  flag_intersection_raw  flag_intersection_strict  raw_only_vs_flag  strict_only_vs_flag
      OPA                 7               121                   71                      2                         2               119                   69
      RIS                 8               124                   76                      6                         6               118                   70
       ML                53                45                   45                     31                        31                14                   14
